In [1]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from glob import glob

SEED = 1

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [2]:
def load_hapt(data_dir, window_size=64, step_size=32):
    raw_dir    = os.path.join(data_dir, 'RawData')
    labels     = np.loadtxt(os.path.join(raw_dir, 'labels.txt'), dtype=int)
    acc_files  = sorted(glob(os.path.join(raw_dir, 'acc_*.txt')))
    gyro_files = sorted(glob(os.path.join(raw_dir, 'gyro_*.txt')))

    signals = {}
    for acc_f, gyro_f in zip(acc_files, gyro_files):
        base    = os.path.basename(acc_f)
        parts   = base.replace('.txt', '').split('_')
        exp_id  = int(parts[1][3:])
        user_id = int(parts[2][4:])
        acc     = np.loadtxt(acc_f)
        gyro    = np.loadtxt(gyro_f)
        signals[(exp_id, user_id)] = np.hstack([acc, gyro])

    train_users = set(range(1, 22))
    X_train, y_train, X_test, y_test = [], [], [], []

    for row in labels:
        exp_id, user_id, act_id, start, end = row
        start -= 1
        key = (exp_id, user_id)
        if key not in signals:
            continue
        seg = signals[key][start:end]
        for i in range(0, len(seg) - window_size + 1, step_size):
            window = seg[i:i + window_size]
            label  = act_id - 1
            if user_id in train_users:
                X_train.append(window)
                y_train.append(label)
            else:
                X_test.append(window)
                y_test.append(label)

    return (np.array(X_train, dtype=np.float32),
            np.array(y_train, dtype=np.int64),
            np.array(X_test,  dtype=np.float32),
            np.array(y_test,  dtype=np.int64))


class HAPTDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return (torch.tensor(self.X[idx], dtype=torch.float32),
                torch.tensor(self.y[idx], dtype=torch.long))

print("load_hapt and HAPTDataset defined.")

load_hapt and HAPTDataset defined.


In [5]:
import os
DATA_DIR = '/content/drive/MyDrive/HAPT'
raw_dir  = os.path.join(DATA_DIR, 'RawData')
X_train, y_train, X_test, y_test = load_hapt(data_dir=DATA_DIR)

print(f"X_train: {X_train.shape}  y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}   y_test:  {y_test.shape}")
print(f"Label range: {y_train.min()} to {y_train.max()}")

X_train: (16063, 64, 6)  y_train: (16063,)
X_test:  (7630, 64, 6)   y_test:  (7630,)
Label range: 0 to 11


In [6]:
 train_loader = DataLoader(HAPTDataset(X_train, y_train),
                          batch_size=64, shuffle=True)
test_loader  = DataLoader(HAPTDataset(X_test, y_test),
                          batch_size=64, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Test  batches: {len(test_loader)}")

Train batches: 251
Test  batches: 120


In [7]:
class GRUClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers  = num_layers
        self.gru = nn.GRU(input_size, hidden_size, num_layers,
                           batch_first=True)
        self.fc  = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0),
                          self.hidden_size).to(x.device)
        out, _ = self.gru(x, h0)
        logit  = self.fc(out[:, -1, :])
        prob   = nn.functional.softmax(logit, dim=1)
        return prob, logit


class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers  = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                             batch_first=True)
        self.fc   = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0),
                          self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0),
                          self.hidden_size).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        logit  = self.fc(out[:, -1, :])
        prob   = nn.functional.softmax(logit, dim=1)
        return prob, logit

print("GRUClassifier and LSTMClassifier defined.")

GRUClassifier and LSTMClassifier defined.


In [8]:
def train_and_evaluate(model, train_loader, test_loader,
                        num_epochs=10, lr=0.001, label=''):
    model     = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(1, num_epochs + 1):
        model.train()
        total_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            _, logit = model(inputs)
            loss = criterion(logit, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"  Epoch {epoch:2d}/{num_epochs} | Loss: {total_loss/len(train_loader):.4f}")

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            _, logit = model(inputs)
            preds    = logit.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

    acc = 100.0 * correct / total
    print(f"  >>> Test Accuracy [{label}]: {acc:.2f}%\n")
    return acc

results = {}

In [9]:
set_seed()
model = GRUClassifier(input_size=6, hidden_size=32,
                       num_layers=1, num_classes=12)
results['GRU h=32 L=1'] = train_and_evaluate(
    model, train_loader, test_loader, label='GRU h=32 L=1'
)

  Epoch  1/10 | Loss: 1.6446
  Epoch  2/10 | Loss: 1.0921
  Epoch  3/10 | Loss: 0.8681
  Epoch  4/10 | Loss: 0.7629
  Epoch  5/10 | Loss: 0.6830
  Epoch  6/10 | Loss: 0.6167
  Epoch  7/10 | Loss: 0.5839
  Epoch  8/10 | Loss: 0.5358
  Epoch  9/10 | Loss: 0.5015
  Epoch 10/10 | Loss: 0.4851
  >>> Test Accuracy [GRU h=32 L=1]: 84.25%



In [10]:
set_seed()
model = GRUClassifier(input_size=6, hidden_size=128,
                       num_layers=1, num_classes=12)
results['GRU h=128 L=1'] = train_and_evaluate(
    model, train_loader, test_loader, label='GRU h=128 L=1'
)

  Epoch  1/10 | Loss: 1.3709
  Epoch  2/10 | Loss: 0.7398
  Epoch  3/10 | Loss: 0.5733
  Epoch  4/10 | Loss: 0.4541
  Epoch  5/10 | Loss: 0.3547
  Epoch  6/10 | Loss: 0.2947
  Epoch  7/10 | Loss: 0.2658
  Epoch  8/10 | Loss: 0.2482
  Epoch  9/10 | Loss: 0.2277
  Epoch 10/10 | Loss: 0.2155
  >>> Test Accuracy [GRU h=128 L=1]: 91.99%



In [11]:
for nl in [2, 3, 4]:
    set_seed()
    model = GRUClassifier(input_size=6, hidden_size=32,
                           num_layers=nl, num_classes=12)
    results[f'GRU h=32 L={nl}'] = train_and_evaluate(
        model, train_loader, test_loader, label=f'GRU h=32 L={nl}'
    )

  Epoch  1/10 | Loss: 1.6102
  Epoch  2/10 | Loss: 0.9131
  Epoch  3/10 | Loss: 0.6282
  Epoch  4/10 | Loss: 0.4723
  Epoch  5/10 | Loss: 0.3867
  Epoch  6/10 | Loss: 0.3364
  Epoch  7/10 | Loss: 0.2996
  Epoch  8/10 | Loss: 0.2764
  Epoch  9/10 | Loss: 0.2618
  Epoch 10/10 | Loss: 0.2415
  >>> Test Accuracy [GRU h=32 L=2]: 88.30%

  Epoch  1/10 | Loss: 1.4503
  Epoch  2/10 | Loss: 0.7664
  Epoch  3/10 | Loss: 0.5587
  Epoch  4/10 | Loss: 0.4427
  Epoch  5/10 | Loss: 0.3796
  Epoch  6/10 | Loss: 0.3248
  Epoch  7/10 | Loss: 0.2887
  Epoch  8/10 | Loss: 0.2612
  Epoch  9/10 | Loss: 0.2486
  Epoch 10/10 | Loss: 0.2323
  >>> Test Accuracy [GRU h=32 L=3]: 87.55%

  Epoch  1/10 | Loss: 1.5706
  Epoch  2/10 | Loss: 0.7826
  Epoch  3/10 | Loss: 0.5421
  Epoch  4/10 | Loss: 0.4236
  Epoch  5/10 | Loss: 0.3493
  Epoch  6/10 | Loss: 0.3004
  Epoch  7/10 | Loss: 0.2604
  Epoch  8/10 | Loss: 0.2359
  Epoch  9/10 | Loss: 0.2321
  Epoch 10/10 | Loss: 0.2348
  >>> Test Accuracy [GRU h=32 L=4]: 90.00%

In [12]:
set_seed()
model = LSTMClassifier(input_size=6, hidden_size=32,
                        num_layers=1, num_classes=12)
results['LSTM h=32 L=1'] = train_and_evaluate(
    model, train_loader, test_loader, label='LSTM h=32 L=1'
)

  Epoch  1/10 | Loss: 1.7127
  Epoch  2/10 | Loss: 1.2014
  Epoch  3/10 | Loss: 0.9598
  Epoch  4/10 | Loss: 0.8814
  Epoch  5/10 | Loss: 0.8342
  Epoch  6/10 | Loss: 0.8535
  Epoch  7/10 | Loss: 0.7139
  Epoch  8/10 | Loss: 0.6864
  Epoch  9/10 | Loss: 0.6524
  Epoch 10/10 | Loss: 0.6730
  >>> Test Accuracy [LSTM h=32 L=1]: 76.50%



In [13]:
print(f"{'Model':<22} | {'Test Accuracy':>12}")
print()
for name, acc in results.items():
    print(f"{name:<22} | {acc:>11.2f}%")

Model                  | Test Accuracy

GRU h=32 L=1           |       84.25%
GRU h=128 L=1          |       91.99%
GRU h=32 L=2           |       88.30%
GRU h=32 L=3           |       87.55%
GRU h=32 L=4           |       90.00%
LSTM h=32 L=1          |       76.50%
